# Demonstration 

## Setup

First, it is necessary to import packages. A `Munch` is a `Dict` that acts like a structure, so we can use dot notation. We'll use this to demonstrate different functions and keep the results organized.

In [1]:
import numpy as np
import astropy.units as u
from munch import Munch

import tellurion as tell
# Orekit-backed functions are available on tell.*

Next we define a specific epoch time `newyear` and a sequence of time steps every 5 minutes for one hour `prop5m1h`

In [2]:
newyear = tell.abstime('2025-01-01T00:00:00')
prop5m1h = np.linspace(5.0*u.minute, 60.0*u.minute, 12) # Step every 5 minutes for an hour

## State and elements


Create the tree structure `demoa` to hold values; `demoa.init` will hold initial values.

In [3]:
demoa = Munch()
demoa.init = Munch()

Create a position, velocity and time, representing the __[orbital state vector](https://en.wikipedia.org/wiki/Orbital_state_vectors)__

In [4]:
demoa.init.pv = [5740.13268349, 3314.06715, 0., -2.75082684, 4.76457184, 5.50165367]
demoa.init.pvt = tell.pvtcart(demoa.init.pv, newyear)
demoa.init.pvt

PositionVelocityT(time=<Time object: scale='utc' format='isot' value=2025-01-01T00:00:00.000>, aux={})

Transform this into a Kepler element set and display it as a dictionary of dimensioned quantities

In [5]:
demoa.init.kep = tell.kepler(demoa.init.pvt)  # Convert PVT to Kepler elements
demoa.init.kep.els.to_dict()

{'inc': <Quantity 45. deg>,
 'argper': <Quantity 0. deg>,
 'raan': <Quantity 30. deg>,
 'ecc': <Quantity 0.0066>,
 'sma': <Quantity 6672.3744 km>,
 'ma': <Quantity -0. deg>}

It is often more useful to know the altitudes of perigee and apogee than just the semimajor axis and inclination; this is accomplished with the function `allplane()`, which includes other useful information such as the mean motion

In [6]:
tell.allplane(demoa.init.kep.els.to_dict())

{'inc': <Quantity 45. deg>,
 'argper': <Quantity 0. deg>,
 'raan': <Quantity 30. deg>,
 'ecc': <Quantity 0.0066>,
 'sma': <Quantity 6672.3744 km>,
 'ma': <Quantity -0. deg>,
 'memo': <Quantity 0.0012 rad / s>,
 'radper': <Quantity 6628.1343 km>,
 'radapo': <Quantity 6716.6145 km>,
 'altper': <Quantity 249.9978 km>,
 'altapo': <Quantity 338.4781 km>}

## Propagation

### Analytical Kepler propagator

In this section, we propagate the previous state `demoa.init.pvt` using an analytical two-body propagator and store the computation in `demoa.propa`. The ephemeris table includes the initial state (this is the default but is shown explicitly here) and shows both the time from epoch and the elapsed time from the previous time step.

In [7]:
demoa.propa = Munch()
demoa.propa.gen = tell.prepare(demoa.init.pvt, 1*u.day, forceenv=tell.kepleranalytic())
demoa.propa.ephem = tell.propagate(demoa.propa.gen, prop5m1h, include_init=True)
demoa.propa.ephem

time,from epoch,elapsed,position,velocity
,,,km,km / s
Time,object,str4,float64[3],float64[3]
2025-01-01T00:00:00.000,0.0s,,5740.133 .. -0.000,-2.750827 .. 5.501654
2025-01-01T00:05:00.000,5min,5min,4581.816 .. 1616.827,-4.891449 .. 5.166423
2025-01-01T00:10:00.000,10min,5min,2865.500 .. 3036.847,-6.432387 .. 4.203822
2025-01-01T00:15:00.000,15min,5min,801.266 .. 4088.442,-7.187958 .. 2.736513
2025-01-01T00:20:00.000,20min,5min,-1359.960 .. 4646.558,-7.074062 .. 0.948419
2025-01-01T00:25:00.000,25min,5min,-3358.333 .. 4647.313,-6.115173 .. -0.941271
2025-01-01T00:30:00.000,30min,5min,-4956.792 .. 4094.285,-4.436283 .. -2.706753
2025-01-01T00:35:00.000,35min,5min,-5968.504 .. 3056.385,-2.243216 .. -4.142427


To get the full vectors, get the `'position'` or `'velocity'` columns explicitly (the suppression of the middle component of the vectors is a limitation of AstroPy that will [fixed soon](https://github.com/astropy/astropy/pull/19123)).

In [8]:
demoa.propa.ephem[3]['position']

<Quantity [ 801.2655, 5183.5367, 4088.4417] km>

Find the altitudes of perigee and apogee; because a two-body propagator is used, they only change due to the slow rotation of the coordinate system.

In [9]:
demoa.propa.altperapo = tell.tselements(demoa.propa.ephem, ["altper","altapo"])
demoa.propa.altperapo

time,altper,altapo
,km,km
Time,float64,float64
2025-01-01T00:00:00.000,249.99783999568226,338.47806177843364
2025-01-01T00:05:00.000,249.99783999568226,338.47806177843734
2025-01-01T00:10:00.000,249.99783999568226,338.47806177843364
2025-01-01T00:15:00.000,249.9978399956841,338.4780617784392
2025-01-01T00:20:00.000,249.99783999568226,338.4780617784299
2025-01-01T00:25:00.000,249.9978399956869,338.47806177843364
2025-01-01T00:30:00.000,249.99783999568132,338.478061778428
2025-01-01T00:35:00.000,249.99783999568783,338.478061778429


### Discrete states and events

*Discrete states* have a discrete number of values (usually two or three), usually non-numeric, that are functions of the state. For example, whether a spacecraft is eclipsed or sunlit. *Events* are transitions between states. In the example of eclipsing, there are three states, designated `u` for umbra (completely eclipsed), `p` for penumbra (partially eclipsed), and `s` for sunlit. Transitions are designated with two letters, for example, `pu` means going from penumbra into umbra. 

In [11]:
demoa.propa.eclipse = Munch()
demoa.propa.eclipse.events = {'altitude': 125.0*u.km, 'eclipse': [True, True], 'visibility': []}
demoa.propa.eclipse.genevpvt = tell.prepare(demoa.init.pvt, 8*u.hour, demoa.propa.eclipse.events, output='pvt')
demoa.propa.eclipse.suntrans = demoa.propa.eclipse.genevpvt['sun transition'].ephemeris()


In [12]:
demoa.propa.eclipse = Munch()
demoa.propa.eclipse.events = {'altitude': 125.0*u.km, 'eclipse': [True, True], 'visibility': []}
demoa.propa.eclipse.genevpvt = tell.prepare(demoa.init.pvt, 8*u.hour, demoa.propa.eclipse.events, forceenv=tell.kepleranalytic(), output='pvt')
demoa.propa.eclipse.suntrans = demoa.propa.eclipse.genevpvt['sun transition'].ephemeris()
demoa.propa.eclipse.suntrans

time,from epoch,elapsed,px,py,pz,eclipse
,,,km,km,km,
Time,object,str13,float64,float64,float64,str2
2025-01-01T00:35:36.378,0.0s,,-6044.865,-137.834,2903.065,up
2025-01-01T00:35:45.143,8.765s,8.765s,-6061.689,-191.094,2865.353,ps
2025-01-01T01:30:00.380,54min 24.002s,54min 15.237s,5803.290,3199.550,-130.753,sp
2025-01-01T01:30:08.972,54min 32.594s,8.592s,5780.974,3241.238,-83.493,pu
2025-01-01T02:06:01.462,1hr 30min 25.084s,35min 52.49s,-6046.690,-143.519,2899.053,up
2025-01-01T02:06:10.229,1hr 30min 33.851s,8.767s,-6063.451,-196.786,2861.304,ps
2025-01-01T03:00:25.518,2hr 24min 49.14s,54min 15.289s,5800.748,3204.370,-125.308,sp
2025-01-01T03:00:34.112,2hr 24min 57.734s,8.594s,5778.361,3246.028,-78.038,pu


The `[True, True]` argument in the `events` dictionary for `'eclipse'` indicates we would like to see both umbra and penumbra events. Notice the time in umbra is around 8 or 9 seconds, as this is a low-earth orbit. 

### Numerical propagator

The previous examples, in `demoa.propa`, used an analytic two-body (Kepler) propagator. To do propagation with perturbations, we usually need a numerical propagator. First, we start with with just the two-body force; propagation results will be saved in `demoa.propn`.

In [13]:
demoa.propn = Munch()

Make the propagation generator, which is necessary to create an ephemeris. A maximum time must be specified; we will make it 1 day.

In [14]:
demoa.propn.gen = tell.prepare(demoa.init.pvt, 1*u.day)

Propagate for 1 hour in 5 minute steps, and include the initial state in the ephemeris table; note the middle (`Y`) value for the position and velocity vectors has been elided, but it is present 

In [15]:
demoa.propn.ephem = tell.propagate(demoa.propn.gen, prop5m1h, True)
demoa.propn.ephem

time,from epoch,elapsed,position,velocity
,,,km,km / s
Time,object,str4,float64[3],float64[3]
2025-01-01T00:00:00.000,0.0s,,5740.133 .. 0.000,-2.750827 .. 5.501654
2025-01-01T00:05:00.000,5min,5min,4581.816 .. 1616.827,-4.891449 .. 5.166423
2025-01-01T00:10:00.000,10min,5min,2865.500 .. 3036.847,-6.432387 .. 4.203822
2025-01-01T00:15:00.000,15min,5min,801.266 .. 4088.442,-7.187958 .. 2.736513
2025-01-01T00:20:00.000,20min,5min,-1359.960 .. 4646.558,-7.074062 .. 0.948419
2025-01-01T00:25:00.000,25min,5min,-3358.333 .. 4647.313,-6.115173 .. -0.941271
2025-01-01T00:30:00.000,30min,5min,-4956.792 .. 4094.285,-4.436283 .. -2.706753
2025-01-01T00:35:00.000,35min,5min,-5968.504 .. 3056.385,-2.243216 .. -4.142427


Let's look at the difference between the and numerical results using the function `magdiff()` that computes the magnitude of the difference between two vectors.

In [16]:
demoa.propa.andiff = tell.magdiff(demoa.propa.ephem['position'], demoa.propn.ephem['position'])
demoa.propa.andiff


<Quantity [0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.] km>

At each of the 13 time steps, the position difference is a maximum of about 5 microns, which is very good agreement.

## Data at single points

This section shows how to get single-point propagation data. In the first computation, propagate to 12 hours with the previously-defined generator, which is within the maximum allowable time of 1 day for this generator.

Find the state at step 5. The function `.pvt()` converts the ephemeris table into a `PositionVelocityT` object, and `.to_dict()` in the last step is to make the elements result easier to read.

In [17]:
pvt5 = demoa.propa.ephem.pvt()[5] # demoa.propa.ephem[5].pvt() would work equally well
pvt5.cartesian.to_dict()

{'position': <Quantity [-3358.3326,  3427.3201,  4647.3126] km>,
 'velocity': <Quantity [-6.1152, -4.6175, -0.9413] km / s>}

Find the state at 45 minutes using the `loc` attribute of tables

In [18]:
demoa.propa.ephem.loc[newyear+45*u.min]

time,from epoch,elapsed,position,velocity
,,,km,km / s
Time,object,str4,float64[3],float64[3]
2025-01-01T00:45:00.000,45min,5min,-5848.977 .. 65.552,2.621940 .. -5.428658


In [19]:
demoa.propa.at45min = demoa.propa.ephem.loc[newyear+45*u.min]
(demoa.propa.at45min['time'], demoa.propa.at45min['position'], demoa.propa.at45min['velocity'])


(<Time object: scale='utc' format='isot' value=2025-01-01T00:45:00.000>,
 <Quantity [-5848.977 , -3301.2156,    65.5519] km>,
 <Quantity [ 2.6219, -4.7547, -5.4287] km / s>)

Or, you could make a `PositionVelocityT` at that time

In [20]:
demoa.propa.at45min.pvt().cartesian

<Quantity ([-5848.977 , -3301.2156,    65.5519], [ 2.6219, -4.7547, -5.4287]) (km, km / s)>

### Time series selection at a single time point

This section shows different ways of getting single points from time series

In [21]:
demoa.tssel = Munch()
demoa.tssel.pvt15m = demoa.propn.ephem[3].pvt() # PVT for 15min by index
demoa.tssel.pvt35m = demoa.propn.ephem.loc['2025-01-01 00:35:00']
demoa.tssel.pvt45m = demoa.propn.ephem.loc[tell.abstime(45*u.min, newyear)].pvt() # PVT for 45min by relative time
demoa.tssel.pvt1h = demoa.propn.ephem[-1].pvt() # PVT at the end of the ephemeris
demoa.tssel.kep1h = tell.kepler(demoa.tssel.pvt1h)  # Convert PVT to Kepler elements


In [23]:
demoa.tssel.pvt15m.cartesian

<Quantity ([ 801.2655, 5183.5367, 4088.4417], [-7.188 , -0.9901,  2.7365]) (km, km / s)>

In [25]:
demoa.tssel.pvt35m

time,from epoch,elapsed,position,velocity
,,,km,km / s
Time,object,str4,float64[3],float64[3]
2025-01-01T00:35:00.000,35min,5min,-5968.504 .. 3056.385,-2.243216 .. -4.142427


In [27]:
demoa.tssel.pvt45m.cartesian

<Quantity ([-5848.977 , -3301.2156,    65.5519], [ 2.6219, -4.7547, -5.4287]) (km, km / s)>

In [28]:
demoa.tssel.pvt1h.time

<Time object: scale='utc' format='isot' value=2025-01-01T01:00:00.000>

In [31]:
demoa.tssel.kep1h.els.to_dict()

{'inc': <Quantity 45. deg>,
 'argper': <Quantity 0. deg>,
 'raan': <Quantity 30. deg>,
 'ecc': <Quantity 0.0066>,
 'sma': <Quantity 6672.3744 km>,
 'ma': <Quantity -121.0685 deg>}